In [ ]:
import warnings
warnings.simplefilter('ignore')

from sklearn import svm
import pickle
import numpy as np
import pandas as pd
from pandas.io.json import json_normalize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import precision_recall_fscore_support
from sklearn.linear_model import LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split


from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import cohen_kappa_score
from sklearn.externals import joblib


from stop_words import get_stop_words
pd.options.mode.chained_assignment = None
import string

import time
import re
import os
import json
import subprocess
import csv
import gzip
from datetime import datetime
import itertools

import matplotlib.pyplot as plt


# Tokenizers
from nltk.tokenize.casual import TweetTokenizer

# Imbalanced classes
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import make_pipeline as make_pipeline_imb
from imblearn.metrics import classification_report_imbalanced
from imblearn.over_sampling import SMOTE, ADASYN

import eli5

# Define Classifier Evaluation Functions
These functions will be used later to test differently balanced tweetsets in order to arrive at the tweetset balance that best supports effective model training (as per F1, Precision, and Recall scores).

In [ ]:
# # from http://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html#sphx-glr-auto-examples-model-selection-plot-confusion-matrix-py
# def plot_confusion_matrix(cm, classes,
#                           normalize=False,
#                           title='Confusion matrix',
#                           cmap=plt.cm.Blues):
#     """
#     This function prints and plots the confusion matrix.
#     Normalization can be applied by setting `normalize=True`.
#     """
#     if normalize:
#         cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
#         print("Normalized confusion matrix")
#     else:
#         print('Confusion matrix, without normalization')

#     print(cm)

#     plt.imshow(cm, interpolation='nearest', cmap=cmap)
#     plt.title(title)
#     plt.colorbar()
#     tick_marks = np.arange(len(classes))
#     plt.xticks(tick_marks, classes, rotation=45)
#     plt.yticks(tick_marks, classes)

#     fmt = '.2f' if normalize else 'd'
#     thresh = cm.max() / 2.
#     for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
#         plt.text(j, i, format(cm[i, j], fmt),
#                  horizontalalignment="center",
#                  color="white" if cm[i, j] > thresh else "black")

#     plt.tight_layout()
#     plt.ylabel('True label')
#     plt.xlabel('Predicted label')
    
    
# def show_conf_matrices(y_test, y_predictions):
#     ## from http://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html#sphx-glr-auto-examples-model-selection-plot-confusion-matrix-py
#     ## compute confusion matrix
#     cnf_matrix = confusion_matrix(y_test, y_predictions)
#     np.set_printoptions(precision=2)

#     ## make class labels
#     y_labels = []
#     for item in y_test:
#         if item not in y_labels:
#             y_labels.append(item)
#         else:
#             continue
    
#     ## plot non-normalized confusion matrix
#     plt.figure(figsize=(8, 6))
#     plot_confusion_matrix(cnf_matrix,classes=y_labels,
#                           title='Confusion matrix, without normalization')

#     ## plot normalized confusion matrix
#     plt.figure(figsize=(8, 6))
#     plot_confusion_matrix(cnf_matrix, classes=y_labels, normalize=True,
#                           title='Normalized confusion matrix')

#     plt.show()

# def eval_classifier(pipeline, X_test, y_test):
#     y_pred = pipeline.predict(X_test)

#     print(classification_report(y_test, y_pred) +  
#           "\nCohen's kappa: " + str(cohen_kappa_score(y_test,y_pred)))
    
#     show_conf_matrices(y_test, y_pred)  

# Set Notebook Parameters
Here we set variables necessary for pre-processing data for classifier training.

In [ ]:
# tokenizer = RegexpTokenizer(r'\w+')
tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True, reduce_len=True) # which tokenizer to use

stop_words_en = get_stop_words('en') # English stoplist
stop_words_sp = get_stop_words('spanish') # Spanish stoplist

stop_words_2 = []

for word in stop_words_en:
    stop_words_2.append(word)
for word in stop_words_sp:
    stop_words_2.append(word)


In [ ]:
def tokenize_func(doc):

    ## remove URLs
    doc = re.sub(r'^(https|http)?:\/\/.*(\r|\n|\b)', '', doc, flags=re.MULTILINE)
    
    ## tokenize
    tokens = tokenizer.tokenize(doc)

    ## remove punctuation from each token
    table = str.maketrans('', '', string.punctuation)
    nopunct_tokens = [w.translate(table) for w in tokens]
    for tok in nopunct_tokens:
        if 'httptco' in tok:
            ind = nopunct_tokens.index(tok)
            del(nopunct_tokens[ind])
        elif 'httpstco' in tok:
            ind = nopunct_tokens.index(tok)
            del(nopunct_tokens[ind])

    ## keep words length 3 or more
    long_tokens = [token for token in nopunct_tokens if len(token) > 2]

    ## remove remaining tokens that are not alphabetic
    alpha_tokens = [word for word in long_tokens if word.isalpha()]

    ## remove stop words from tokens
    stopped_tokens = [i for i in alpha_tokens if not i in stop_words_2]
        
    return stopped_tokens

def replace_urls(text):
    text_clean = re.sub(r'^(https|http)?:\/\/.*(\r|\n|\b)', '', text, flags=re.MULTILINE)
    return text_clean

# Get Data

Russell's raw data is not publicly available at this time (we received Russell's raw data via private email). As such, though we include all code we used to prepare the data to build and test our supervised model, we have commented out those lines which require raw datasets. 
    
The processed data that we used to train and test the model can be found at:

    '../../../../data/russell/russell_processed_0.9.csv.gz'

In [ ]:
# data_file = 'RAWDATA_russell'
# tweets = pd.read_csv(data_file, dtype=str, compression='gzip')

# Process Data

In [ ]:
# def process_data(tweets):
    
#     ## rename columns
#     tweets['new_topic_1'] = tweets['new_topic_1'].astype(str)
#     tweets.rename(index=str, columns={'Topic Code': 'topic_code'})

#     col_names = list(tweets.columns.values)

#     ## uncomment to display tweets dataframe dimensions
#     # tweets.shape
    
#     ## remove retweets in training set
#     tweets['original'] = np.where(tweets.text.str.startswith('RT'), 0, 1)

#     original_tweets = tweets[tweets['original'] == 1]

#     ## recast datatypes for relevant columns
#     original_tweets['topic_code'] = original_tweets['Topic Code'].apply(str)
#     original_tweets['datetime'] = pd.to_datetime(original_tweets['Date'])
#     original_tweets['date'] = original_tweets['datetime'].dt.date
#     original_tweets['year'] = original_tweets['datetime'].dt.year
#     original_tweets['month'] = original_tweets['datetime'].dt.month
    
#     ## create new column with tweet text sans urls
#     original_tweets['text_no_urls'] = original_tweets['text'].apply(replace_urls)
    
#     ## uncomment to display column names
#     # original_tweets.columns.values
    
#     original_tweets = original_tweets.drop(['Unnamed: 0', 'KEYID', 'State', 'Name', 'text', 'Date', 
#                      'Unique_ID', 'Topic Code', 'Policy Content', 'Oversight', 
#                      'Partisanship', 'Elections', 'CreditClaim', 'Constituent Meeting', 
#                      'Constituent Issues/Details', 'Sports', 'Holidays/Anniversaries', 
#                      'Press Release', 'Photos or Videos', 'Media Appearance', 'Media/Report', 
#                      'Inst. Action', 'CODA', 'Speech', 'Link#', 'Boston', 'Link', 'DirectCommCon', 
#                      'DirectCommElite', 'Other', 'POSITIVE', 'NEGATIVE', 'Negative', 'Positive', 
#                      'Nuetral', 'Sentiment', 'RT', 'Retweets', 'Hashtags', 'Constituent', 
#                      'Unique_ID_two', 'text_two', 'unique_id', 'unique_id_two', 'new_topic_2', 'original', 
#                      'topic_code', 'datetime', 'date', 'year', 'month'], axis=1)
    
#     return original_tweets

In [ ]:
# original_tweets = process_data(tweets)

## Balance Tweetset
Non-policy tweets are disproportionately represented in the dataset in relation to policy tweets. As such, this code functions to balance the tweetset in such way that our classifier may more effectively learn classification patterns

In [ ]:
# def balance_tweetset(original_tweets):
    
#     ## this code section tests different fractions of non-policy tweets to keep in increments of 10%.
#     ## need to take this df of policy tweets, and merge back (or rather concatenate back) with a random 
#     ## selection of those Tweets classified as '0'

#     fractions = [.1,.2,.3,.4,.5,.6,.7,.8,.9]
#     fractions_dfs = {}

#     for f in fractions:
#         policy_filter = original_tweets.copy()
#         policy_filter = policy_filter.drop(policy_filter.query('new_topic_1 == "0"' ).sample(frac=f).index)
#         policy_0 = policy_filter[policy_filter.new_topic_1=='0']
#         policy_filter['text_tokenized'] = [tokenize_func(n) for n in policy_filter.text]
            
#         fractions_dfs[f] = policy_filter
        
#     return fractions_dfs

In [ ]:
# fractions_dfs = balance_tweetset(original_tweets)

In [ ]:
## output best performing balance to csv, uncomment to use
# fractions_dfs[0.9].to_csv('../../../../data/russell/russell_processed_0.9.csv.gz', 
#                           compression='gzip', index=False)

## Make Training and Test Sets
Here each tweet is its own document. We include code to test all tweetset balances. We provide our final processed Russell dataset, which inludes 90% non-policy tweets. As such, if attempting to reproduce our work, the code that follows will display results for this tweetset balance only.

In [ ]:
## load data here if attempting to reproduce the below code, this is Russell's processed data
data_file = 'russell_processed_0.9.csv.gz'
tweets = pd.read_csv(data_file, dtype=str, compression='gzip')
fractions_dfs = {0.9:tweets}

In [ ]:
def make_training_docs(df):
    
    x = np.array(df.text_no_urls)
    y = np.asarray(df['new_topic_1'])

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.10, random_state=42)
    
    return x_train, x_test, y_train, y_test

In [ ]:
def make_dict_balanced_training_docs(fractions_dfs):

    ## creating a dict of all of the different fractions of non-policy tweets to include in train and test sets
    train_test = {}

    for k,v in fractions_dfs.items():
        x_train, x_test, y_train, y_test = make_training_docs(v)
        train_test[k] = x_train, x_test, y_train, y_test
        # print("Using " + str(len(x_train)) + " original tweets to TRAIN the model. This excludes Retweets.")
        # print("Using " + str(len(x_test)) + " original tweets to TEST the model. This excludes Retweets.")
    
    return train_test

In [ ]:
train_test = make_dict_balanced_training_docs(fractions_dfs)

# CountVectorizer

In [ ]:
## Turn train and test sets into vectors using CountVectorizer (Bag of Words)
vec = CountVectorizer(tokenizer=tokenize_func)

# Test Classifiers With Different Tweetsets

When attempting to run a logistic regression classifier on the child class (original) topic labeling system Dr. Russell used (i.e. 1001 for topic 10, subtopic 1), received the following error: "The least populated class in y has only 1 members, which is too few. The minimum number of members in any class cannot be less than n_splits=3." This could also be what affected initial Naive Bayes performance. As such, new models of each type were created to predict only the PARENT class in Dr. Russell's data (i.e. 10 for topic 10). This parent class was added as an additional column to Dr. Russell's data.

The overall best performing model predicting parent classes (F1 = 0.791, Cohen's Kappa with human labels = 0.769) was a logistic regression including 90% of non-policy tweets (accessed by calling fractions_dfs[0.9]).

## Dummy Classifier

In [ ]:
%%time
dmy = DummyClassifier(strategy='stratified')

pipeline = make_pipeline(vec,
                        dmy)

for k,v in train_test.items():
    print("WITH " + str(k*100) + "% 0S REMOVED")
    pipeline.fit(v[0], v[2])
    ## uncomment to save model
    # joblib.dump(pipeline, '../../../../models/best/d/dmy_' + str(k) + '.pkl')
    eval_classifier(pipeline, v[1], v[3])

## Naive Bayes Classifier

In [ ]:
%%time
nb = MultinomialNB()

pipeline = make_pipeline(vec,
                        nb)

for k,v in train_test.items():
    print("WITH " + str(k*100) + "% 0S REMOVED")
    pipeline.fit(v[0], v[2])
#     joblib.dump(pipeline, '../../../../models/best/nb/nb_' + str(k) + '.pkl')
    eval_classifier(pipeline, v[1], v[3])

## Logistic Regression Classifier

In [ ]:
%%time
lr = LogisticRegressionCV(multi_class='ovr')

pipeline = make_pipeline(vec,
                        lr)

# v[0] = x_train, v[2] = y_train, v[1] = x_test, v[3] = y_test
for k,v in train_test.items():
    print("WITH " + str(k*100) + "% 0S REMOVED")
    pipeline.fit(v[0], v[2])
#     joblib.dump(pipeline, '../../../../models/best/lr/lr_' + str(k) + '.pkl')
    eval_classifier(pipeline, v[1], v[3])

In [ ]:
%%time
lr = LogisticRegressionCV(multi_class='ovr')

pipeline = make_pipeline(vec,
                        lr)

## reminder: v[0] = x_train, v[2] = y_train, v[1] = x_test, v[3] = y_test
for k,v in train_test.items():
    if k == 0.9:
        print("WITH " + str(k*100) + "% 0S REMOVED")
        pipeline.fit(v[0], v[2])
        ## uncomment to save model
        # joblib.dump(pipeline, '../../../../models/best/lr/lr_' + str(k) + '.pkl')
        eval_classifier(pipeline, v[1], v[3])
    else:
        continue